# Project 4: Air Quality Insights Dashboard

**Student:** Yashasvi Kumar Singh  
**Dataset:** `04_air_quality.csv`

This notebook covers the data cleaning part of the project. I check the dataset, handle missing PM2.5 values, convert the date column and check for unusual values.

## 1. Import libraries and load the dataset

In [1]:
import numpy as np
import pandas as pd

# Load the dataset.
file_path = "04_air_quality.csv"
df = pd.read_csv(file_path)

## 2. Inspect the dataset

First, I check the rows, columns and data types.

In [2]:
print("First 5 rows:")
print(df.head())

print("\nShape (rows, columns):")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nDataFrame information:")
df.info()

First 5 rows:
         Date        City   PM25   PM10   NO2  Temperature_C  Humidity_Percent
0  2025-01-01       Delhi   28.6  111.6  47.4           15.4                26
1  2025-01-02      Jaipur   59.8  160.7  51.0           30.4                49
2  2025-01-03  Chandigarh  105.0  157.6  40.5           18.4                68
3  2025-01-04  Chandigarh   86.5   55.6  28.9           26.5                33
4  2025-01-05  Chandigarh  134.6  154.1  47.5           51.2                88

Shape (rows, columns):
(600, 7)

Columns:
Index(['Date', 'City', 'PM25', 'PM10', 'NO2', 'Temperature_C',
       'Humidity_Percent'],
      dtype='object')

DataFrame information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              600 non-null    object 
 1   City              600 non-null    object 
 2   PM25              585 non-null    f

**What I found:**
- There are 600 rows and 7 columns.
- `Date` is stored as text.
- `PM25` has 15 missing values.

In [3]:
# Summary statistics.
print(df.describe())

             PM25        PM10         NO2  Temperature_C  Humidity_Percent
count  585.000000  600.000000  600.000000     600.000000        600.000000
mean    82.628376  147.568667   35.657500      26.504667         55.788333
std     35.716442   49.529069   12.073816       8.116998         21.477562
min      5.000000   10.000000    2.000000       2.300000         20.000000
25%     58.600000  114.025000   27.975000      20.600000         37.000000
50%     82.100000  146.550000   35.750000      26.400000         56.000000
75%    105.900000  181.725000   43.625000      31.600000         73.000000
max    192.300000  302.300000   72.500000      60.200000         95.000000


## 3. Check missing values

In [4]:
# Check missing values.
print(df.isnull().sum())

Date                 0
City                 0
PM25                15
PM10                 0
NO2                  0
Temperature_C        0
Humidity_Percent     0
dtype: int64


Only `PM25` has missing values. I will check which cities they belong to before filling them.

In [5]:
# Show rows with missing PM25.
missing_rows = df[df["PM25"].isnull()]
print(missing_rows)

# Count missing values by city.
print("\nMissing PM25 values per city:")
print(missing_rows["City"].value_counts())

           Date        City  PM25   PM10   NO2  Temperature_C  Humidity_Percent
65   2025-03-07  Chandigarh   NaN  178.5  22.2           21.2                88
133  2025-05-14       Delhi   NaN   58.1  40.9           22.2                66
145  2025-05-26      Jaipur   NaN  135.7  30.6           17.8                50
207  2025-07-27       Delhi   NaN  184.5  35.3           33.6                73
267  2025-09-25     Lucknow   NaN   83.0  29.8           32.6                23
332  2025-11-29      Jaipur   NaN  130.7  44.2           28.1                31
402  2026-02-07     Lucknow   NaN   80.0  37.5           17.2                59
413  2026-02-18  Chandigarh   NaN  128.4  19.2           40.5                63
456  2026-04-02     Lucknow   NaN   99.8  48.5           26.2                33
470  2026-04-16     Lucknow   NaN  112.2  35.5           23.1                66
530  2026-06-15     Lucknow   NaN  158.1  38.4           38.1                77
537  2026-06-22     Lucknow   NaN  124.6

## 4. Check duplicate records

In [6]:
# Check duplicate rows.
print("Duplicate rows:", df.duplicated().sum())

# Check repeated city-date pairs.
print("Duplicate City + Date pairs:", df.duplicated(subset=["Date", "City"]).sum())

Duplicate rows: 0
Duplicate City + Date pairs: 0


There are no complete duplicate rows or repeated city-date pairs, so no rows need to be removed.

## 5. Handle missing PM2.5 values

There are 15 missing values in `PM25`. I am filling them with the median for the same city. I chose the median because it is less affected by very high or low readings. This also keeps the city values separate instead of using one median for the whole dataset.

In [7]:
# Keep the original data unchanged.
clean_df = df.copy()

# Mark values that need filling.
clean_df["PM25_Imputed"] = clean_df["PM25"].isnull()

# Check the median for each city.
print("Median PM25 per city:")
print(clean_df.groupby("City")["PM25"].median())

Median PM25 per city:
City
Chandigarh    82.30
Delhi         80.70
Jaipur        81.85
Lucknow       82.40
Name: PM25, dtype: float64


In [8]:
# Fill missing PM25 values with the city median.
city_median = clean_df.groupby("City")["PM25"].transform("median")
clean_df["PM25"] = clean_df["PM25"].fillna(city_median)

# Check the result.
print("Missing PM25 before:", df["PM25"].isnull().sum())
print("Missing PM25 after :", clean_df["PM25"].isnull().sum())
print("Rows filled        :", clean_df["PM25_Imputed"].sum())

Missing PM25 before: 15
Missing PM25 after : 0
Rows filled        : 15


In [9]:
# Compare PM25 before and after filling.
print("Before filling:")
print(df["PM25"].describe().round(2))

print("\nAfter filling:")
print(clean_df["PM25"].describe().round(2))

Before filling:
count    585.00
mean      82.63
std       35.72
min        5.00
25%       58.60
50%       82.10
75%      105.90
max      192.30
Name: PM25, dtype: float64

After filling:
count    600.00
mean      82.61
std       35.27
min        5.00
25%       60.18
50%       82.25
75%      105.32
max      192.30
Name: PM25, dtype: float64


The mean and standard deviation changed only slightly after filling the missing values. I kept the `PM25_Imputed` column to show which values were filled.

## 6. Convert the date column

The `Date` column is stored as text, so I convert it to datetime. I also add year, month and year-month columns for the later analysis.

In [10]:
# Convert Date to datetime.
clean_df["Date"] = pd.to_datetime(clean_df["Date"])

# Add year and month columns.
clean_df["Year"] = clean_df["Date"].dt.year
clean_df["Month"] = clean_df["Date"].dt.month
clean_df["YearMonth"] = clean_df["Date"].dt.strftime("%Y-%m")

print(clean_df[["Date", "City", "Year", "Month", "YearMonth"]].head())
print("\nDate column type:", clean_df["Date"].dtype)
print("Earliest date:", clean_df["Date"].min())
print("Latest date  :", clean_df["Date"].max())

        Date        City  Year  Month YearMonth
0 2025-01-01       Delhi  2025      1   2025-01
1 2025-01-02      Jaipur  2025      1   2025-01
2 2025-01-03  Chandigarh  2025      1   2025-01
3 2025-01-04  Chandigarh  2025      1   2025-01
4 2025-01-05  Chandigarh  2025      1   2025-01

Date column type: datetime64[ns]
Earliest date: 2025-01-01 00:00:00
Latest date  : 2026-08-23 00:00:00


In [11]:
# Count records by year and month.
print(clean_df["YearMonth"].value_counts().sort_index())

YearMonth
2025-01    31
2025-02    28
2025-03    31
2025-04    30
2025-05    31
2025-06    30
2025-07    31
2025-08    31
2025-09    30
2025-10    31
2025-11    30
2025-12    31
2026-01    31
2026-02    28
2026-03    31
2026-04    30
2026-05    31
2026-06    30
2026-07    31
2026-08    23
Name: count, dtype: int64


## 7. Check unusual values

I checked for rows where `PM10` is lower than `PM25` and temperatures above 45°C. I only flag these rows for review and leave the original values unchanged.

In [12]:
# Flag rows where PM10 is lower than PM25.
clean_df["PM10_Lower_Than_PM25"] = np.where(
    (clean_df["PM10"] < clean_df["PM25"]) & (~clean_df["PM25_Imputed"]),
    1,
    0
)

# Flag temperatures above 45°C.
clean_df["High_Temp_Flag"] = np.where(clean_df["Temperature_C"] > 45, 1, 0)

print("Rows where PM10 < PM25:", clean_df["PM10_Lower_Than_PM25"].sum())
print("Rows with temperature > 45 C:", clean_df["High_Temp_Flag"].sum())
print("Highest temperature:", clean_df["Temperature_C"].max())

Rows where PM10 < PM25: 83
Rows with temperature > 45 C: 7
Highest temperature: 60.2


In [13]:
# Show rows with high temperatures.
high_temp_rows = clean_df[clean_df["High_Temp_Flag"] == 1]
print(high_temp_rows[["Date", "City", "Temperature_C"]])

          Date        City  Temperature_C
4   2025-01-05  Chandigarh           51.2
18  2025-01-19      Jaipur           46.4
103 2025-04-14  Chandigarh           52.6
261 2025-09-19      Jaipur           46.0
421 2026-02-26       Delhi           49.1
515 2026-05-31     Lucknow           47.4
531 2026-06-16     Lucknow           60.2


These values are only flagged because I cannot tell from the dataset what the correct values should be.

## 8. Check the cleaned dataset

In [14]:
print("Shape before cleaning:", df.shape)
print("Shape after cleaning :", clean_df.shape)

print("\nMissing values after cleaning:")
print(clean_df.isnull().sum())

print("\nCleaned DataFrame information:")
clean_df.info()

Shape before cleaning: (600, 7)
Shape after cleaning : (600, 13)

Missing values after cleaning:
Date                    0
City                    0
PM25                    0
PM10                    0
NO2                     0
Temperature_C           0
Humidity_Percent        0
PM25_Imputed            0
Year                    0
Month                   0
YearMonth               0
PM10_Lower_Than_PM25    0
High_Temp_Flag          0
dtype: int64

Cleaned DataFrame information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  600 non-null    datetime64[ns]
 1   City                  600 non-null    object        
 2   PM25                  600 non-null    float64       
 3   PM10                  600 non-null    float64       
 4   NO2                   600 non-null    float64       
 5   Temp

In [15]:
# Preview the cleaned data.
print(clean_df.head())

        Date        City   PM25  ...  YearMonth  PM10_Lower_Than_PM25  High_Temp_Flag
0 2025-01-01       Delhi   28.6  ...    2025-01                     0               0
1 2025-01-02      Jaipur   59.8  ...    2025-01                     0               0
2 2025-01-03  Chandigarh  105.0  ...    2025-01                     0               0
3 2025-01-04  Chandigarh   86.5  ...    2025-01                     1               0
4 2025-01-05  Chandigarh  134.6  ...    2025-01                     0               1

[5 rows x 13 columns]


## 9. Save the cleaned dataset

In [16]:
# Save the cleaned dataset.
output_file = "04_air_quality_cleaned.csv"
clean_df.to_csv(output_file, index=False)
print("Saved", output_file, "with shape", clean_df.shape)

Saved 04_air_quality_cleaned.csv with shape (600, 13)


## Summary

- Filled the 15 missing `PM25` values using each city's median.
- No duplicate rows were found.
- Converted `Date` to datetime and added year/month columns.
- Added flags for possible unusual values.
- Saved the cleaned data as `04_air_quality_cleaned.csv`.

All 600 rows were kept.